# Model 1: Zero-Shot Inference
**Modell:** `Qwen2.5-1.5B-Instruct` (lokal, kein API)  
**Methode:** Zero-Shot — direkte Antworten ohne Beispiele oder externe Dokumente


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "tqdm"])
print("✓ Pakete bereit")

In [ ]:
import os, csv, torch
import pandas as pd
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

DATASET = "../../dataset_clean.csv"
OUT     = "../results/results_model1_inference.csv"
os.makedirs("../results", exist_ok=True)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

def load_questions():
    with open(DATASET, newline="", encoding="utf-8-sig") as f:
        return [{"id": r["id"], "prompt": r["prompt"]} for r in csv.DictReader(f)]

questions = load_questions()
print(f"{len(questions)} Fragen geladen.")
pd.DataFrame(questions).head(3)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Lade {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device)
model.eval()
print("✓ Modell geladen")

In [ ]:
SYSTEM = (
    "Du bist ein Experte für österreichisches Steuerrecht. "
    "Beantworte die Frage präzise und fachlich korrekt auf Deutsch. "
    "Beziehe dich auf relevante Gesetze (EStG, KStG, UStG, BAO) wenn zutreffend. "
    "Halte die Antwort kurz und prägnant."
)

def infer(question):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300, temperature=0.2,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Schnelltest
print(infer("Was ist der Körperschaftsteuersatz in Österreich?"))

In [ ]:
results = []
for q in tqdm(questions, desc="Model 1"):
    results.append({"id": q["id"], "answer": infer(q["prompt"])})

df = pd.DataFrame(results)
df.to_csv(OUT, index=False)
print(f"✓ Gespeichert: {OUT}  ({len(df)} Zeilen)")
df.head(3)